# Vibe Runner — Multi-Industry Pipeline Orchestrator

**Batch orchestration of the Vibe Modelling Agent across multiple industries.**

The Vibe Runner reads a batch input JSON containing widget values and one or more business/industry definitions, then for each industry orchestrates a 4-task Databricks job pipeline:

1. **Generate an ECM** (Expanded Coverage Model) — full-scope base model
2. **Install the ECM** into a dedicated Unity Catalog catalog
3. **Shrink the ECM** into an MVM (Minimum Viable Model)
4. **Install the MVM** into a dedicated Unity Catalog catalog

After all tasks succeed, the runner copies volume artifacts to a local folder and drops the staging catalog.

---

## Widgets

| # | Widget | Type | Default | Purpose |
|:--|:---|:---:|:---|:---|
| 1 | **Industry Context File** | Text | `/Workspace/Users/<your-user>/vibe-modelling/industries.json` | Path to the batch input JSON |
| 2 | **Dry Run** | Dropdown | `yes` | `yes` = simulated pipeline, `no` = real execution |
| 3 | **Ping Interval** | Dropdown | `1m` | Status logging frequency: `10s` to `15m` |

**Auto-discovered:** Agent notebook path (resolved relative to this notebook) and output folder (`../models`).

---

## Task DAG

```
ECM Generate (task 1)
|-- ECM Install (task 2)   [depends on task 1]
+-- MVM Shrink (task 3)    [depends on task 1]
    +-- MVM Install (task 4)   [depends on task 3]
```

Tasks 2 and 3 run in parallel after Task 1 completes. Task 4 runs after Task 3 finishes.

---

## Quick Start

1. Upload your `industries.json` batch file to Databricks workspace
2. Set the **Industry Context File** widget to point to it
3. Optionally set **Dry Run** to `no` for real execution
4. Click **Run All**

See `runner/runner-guide.md` for full documentation.

In [0]:
import json
import os
import posixpath
import re
import time
import uuid
import shutil
from datetime import datetime

try:
    from databricks.sdk import WorkspaceClient
    from databricks.sdk.service import jobs, catalog as catalog_service
    from databricks.sdk.service.workspace import ImportFormat, Language
    from pyspark.sql import SparkSession
except ImportError:
    pass
 
POLL_INTERVAL_SECONDS = 30
JOB_TIMEOUT_SECONDS = 43200
MAX_SIGNED_INT64 = 9223372036854775807

_TAG_SAFE_RE = re.compile(r'[^A-Za-z0-9._-]')


def _sanitize_tag(value):
    s = _TAG_SAFE_RE.sub('_', str(value))
    s = re.sub(r'_+', '_', s)
    s = s.strip('_').strip('.').strip('-')
    return s


def build_job_tags(business_name, operation, notebook_path, model_scope="ecm", version="1",
                   session_id="", launcher_source="Vibe_Modelling_Notebook"):
    ms = model_scope.lower()
    scope_short = "ecm" if "ecm" in ms else "mvm"
    nb_filename = notebook_path.rsplit("/", 1)[-1] if notebook_path else "unknown"
    sid = session_id or generate_session_id()
    return {
        _sanitize_tag(k): _sanitize_tag(v)
        for k, v in {
            "dbx_vibe_modelling_launcher_source": launcher_source,
            "dbx_vibe_modelling_business": business_name,
            "dbx_vibe_modelling_model": f"{scope_short}_v{version}",
            "dbx_vibe_modelling_operation": operation,
            "dbx_vibe_modelling_notebook": nb_filename,
            "dbx_vibe_modelling_session_id": sid,
            "dbx_vibe_modelling_domains": "0",
            "dbx_vibe_modelling_products": "0",
            "dbx_vibe_modelling_attributes": "0",
            "dbx_vibe_modelling_foreign_keys": "0",
            "dbx_vibe_modelling_tags": "0",
            "dbx_vibe_modelling_metrics": "0",
        }.items()
    }


def find_or_create_job(w, job_name, notebook_path, params, job_tags, task_configs=None,
                       timeout_seconds=JOB_TIMEOUT_SECONDS):
    if task_configs:
        task_list = []
        for tc in task_configs:
            task = jobs.Task(
                task_key=tc["task_key"],
                notebook_task=jobs.NotebookTask(
                    notebook_path=tc.get("notebook_path", notebook_path),
                    base_parameters=tc.get("params", params),
                ),
                timeout_seconds=tc.get("timeout_seconds", timeout_seconds),
                max_retries=tc.get("max_retries", 0),
            )
            if "depends_on" in tc:
                task.depends_on = [jobs.TaskDependency(task_key=dep) for dep in tc["depends_on"]]
            task_list.append(task)
    else:
        task_list = [
            jobs.Task(
                task_key="vibe_task",
                notebook_task=jobs.NotebookTask(
                    notebook_path=notebook_path,
                    base_parameters=params,
                ),
                timeout_seconds=timeout_seconds,
                max_retries=0,
            )
        ]

    existing_job_id = None
    reused = False
    try:
        for j in w.jobs.list(name=job_name):
            if j.settings and j.settings.name == job_name:
                existing_job_id = j.job_id
                break
    except Exception:
        pass

    if existing_job_id:
        w.jobs.reset(
            job_id=existing_job_id,
            new_settings=jobs.JobSettings(
                name=job_name,
                tags=job_tags,
                tasks=task_list,
            ),
        )
        job_id = existing_job_id
        reused = True
        log(f"    Reused existing job: {job_name} (job_id={job_id})")
    else:
        created = w.jobs.create(
            name=job_name,
            tags=job_tags,
            tasks=task_list,
        )
        job_id = created.job_id
        log(f"    Created new job: {job_name} (job_id={job_id})")

    tag_str = ", ".join(f"{k}={v}" for k, v in job_tags.items())
    log(f"    Tags: {tag_str}")

    run = w.jobs.run_now(job_id=job_id)
    run_id = run.run_id
    log(f"    Triggered run: run_id={run_id}")

    return job_id, run_id, reused


def log(msg=""):
    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    if msg:
        print(f"[{ts}] {msg}")
    else:
        print()


def generate_session_id():
    return str((uuid.uuid4().int >> 64) & MAX_SIGNED_INT64)

WIDGET_VALUE_KEYS = [
    "business_domains", "org_divisions",
    "cataloging_style", "catalog_prefix", "catalog_suffix",
    "generate_samples", "naming_convention", "primary_key_suffix",
    "schema_prefix", "schema_suffix", "tag_prefix", "tag_suffix",
    "table_id_type", "boolean_format", "date_format", "timestamp_format",
    "classification_levels", "housekeeping_columns", "history_tracking_columns",
]

RUNNER_ASCII_ART = r"""
    ╔══════════════════════════════════════════════════════════════════════════════╗
    ║   __     __ _  _             ____                                          ║
    ║   \ \   / /(_)| |__    ___  |  _ \  _   _  _ __   _ __    ___  _ __        ║
    ║    \ \ / / | || '_ \  / _ \ | |_) || | | || '_ \ | '_ \  / _ \| '__|       ║
    ║     \ V /  | || |_) ||  __/ |  _ < | |_| || | | || | | ||  __/| |          ║
    ║      \_/   |_||_.__/  \___| |_| \_\ \__,_||_| |_||_| |_| \___||_|          ║
    ║                                                                            ║
    ║   Multi-Industry ECM/MVM Pipeline Orchestrator                              ║
    ╚══════════════════════════════════════════════════════════════════════════════╝
"""



def sanitize_name(name, strip_stop_words=True):
    if not name:
        return "unnamed_model"
    s = str(name).lower()
    if strip_stop_words:
        s = re.sub(r'[(),.&]', ' ', s)
        word_stop_words = ['and', 'or', 'with', 'by', 'of', 'for', 'the', 'a', 'an', 'in', 'on', 'at',
                           'services', 'solutions', 'business', 'inc', 'llc', 'corp']
        pattern = r'\b(' + '|'.join(re.escape(w) for w in word_stop_words) + r')\b'
        s = re.sub(pattern, '', s)
    s = re.sub(r'[^a-z0-9_]', '_', s)
    s = re.sub(r'_+', '_', s).strip('_')
    if s and s[0].isdigit():
        s = '_' + s
    if not s or s == '_':
        first_word = re.sub(r'[^a-z0-9]', '', str(name).lower().split()[0] if str(name).split() else '')
        return first_word if first_word else "unnamed_model"
    return s


def create_widgets():
    dbutils.widgets.removeAll()
    dbutils.widgets.text("business_context", "/Workspace/Users/<your-user>/vibe-modelling/industries.json", "1. Industry Context File")
    dbutils.widgets.dropdown("dry_run", "yes", ["yes", "no"], "2. Dry Run")
    dbutils.widgets.dropdown("ping_interval", "1m", ["10s", "20s", "30s", "1m", "2m", "5m", "10m", "15m"], "3. Ping Interval")


def read_widgets():
    business_context_path = dbutils.widgets.get("business_context").strip()
    folder_path = "./../models"
    dry_run = dbutils.widgets.get("dry_run").strip().lower() == "yes"
    ping_raw = dbutils.widgets.get("ping_interval").strip().lower()
    try:
        if ping_raw.endswith("s"):
            ping_interval = int(ping_raw[:-1])
        elif ping_raw.endswith("m"):
            ping_interval = int(ping_raw[:-1]) * 60
        else:
            ping_interval = int(ping_raw) * 60
    except (ValueError, TypeError):
        ping_interval = 60
    ping_label = f"{ping_interval}s" if ping_interval < 60 else f"{ping_interval // 60}m"

    try:
        runner_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        runner_dir = "/".join(runner_path.split("/")[:-1])
        notebook_path = posixpath.normpath(posixpath.join(runner_dir, "./../agent/dbx_vibe_modelling_agent"))
    except Exception:
        notebook_path = "./../agent/dbx_vibe_modelling_agent"

    log(f"  Industry Context : {business_context_path}")
    log(f"  Agent Notebook   : {notebook_path}")
    log(f"  Folder Path      : {folder_path}")
    log(f"  Dry Run          : {'YES' if dry_run else 'NO'}")
    log(f"  Ping Interval    : {ping_label}")
    log()

    return {
        "business_context_path": business_context_path,
        "notebook_path": notebook_path,
        "folder_path": folder_path,
        "dry_run": dry_run,
        "ping_interval": ping_interval,
    }


def load_business_context(json_path):
    abs_path = os.path.abspath(json_path)
    try:
        with open(abs_path, 'r') as f:
            raw_content = f.read()
    except FileNotFoundError:
        raise FileNotFoundError(f"Industry context file not found: {abs_path}")

    try:
        data = json.loads(raw_content)
    except json.JSONDecodeError as e:
        lines = raw_content.split('\n')
        error_line_idx = e.lineno - 1 if e.lineno else 0
        context_start = max(0, error_line_idx - 2)
        context_end = min(len(lines), error_line_idx + 3)
        context_lines = []
        for i in range(context_start, context_end):
            marker = " >>>" if i == error_line_idx else "    "
            context_lines.append(f"{marker} L{i+1}: {lines[i]}")
        context_block = "\n".join(context_lines)
        raise ValueError(
            f"Invalid JSON in industry context file: {abs_path}\n"
            f"Error: {e.msg} at line {e.lineno}, column {e.colno} (char {e.pos})\n"
            f"File size: {len(raw_content):,} bytes, {len(lines)} lines\n"
            f"\nContext around the error:\n{context_block}\n\n"
            f"Fix: Ensure the file at '{abs_path}' contains valid JSON. "
            f"You can validate it with: python3 -c \"import json; json.load(open('{json_path}'))\""
        ) from e

    widget_values = data.get("widget_values", {})
    businesses = data.get("businesses", [])

    if not businesses:
        raise ValueError("No industries found in context file")

    missing_keys = [k for k in WIDGET_VALUE_KEYS if k not in widget_values]
    if missing_keys:
        raise ValueError(
            f"Industry context file is missing required widget_values keys: {missing_keys}\n"
            f"All of these must be present in the 'widget_values' section: {WIDGET_VALUE_KEYS}"
        )

    log(f"  Loaded {len(businesses)} industries from {abs_path}")
    log(f"  Widget values: {len(widget_values)} parameters")
    for k in WIDGET_VALUE_KEYS:
        log(f"    {k}: {widget_values[k]}")

    vibes_overrides = [b["name"] for b in businesses if b.get("model_vibes")]
    if vibes_overrides:
        log(f"  Business-level model_vibes overrides: {', '.join(vibes_overrides)}")
    return widget_values, businesses


def build_notebook_params(widget_values, business, operation, deployment_catalog,
                          data_model_scopes, context_file="", model_version="",
                          generate_samples=None):
    params = {}
    for key in WIDGET_VALUE_KEYS:
        params[key] = widget_values[key]

    params["business_name"] = business["name"]
    params["business_description"] = business.get("description", "")
    params["operation"] = operation
    params["model_version"] = model_version
    params["data_model_scopes"] = data_model_scopes
    params["deployment_catalog"] = deployment_catalog
    params["context_file"] = context_file
    params["vibe_session_id"] = generate_session_id()

    biz_overrides = business.get("widget_values", {})
    for k, v in biz_overrides.items():
        params[k] = v

    params["model_vibes"] = business.get("model_vibes", "")

    if generate_samples is not None:
        params["generate_samples"] = str(generate_samples)

    return params


def ensure_staging_catalog(spark, catalog_name):
    try:
        spark.sql(f"DROP CATALOG IF EXISTS `{catalog_name}` CASCADE")
        spark.sql(f"CREATE CATALOG `{catalog_name}`")
        log(f"    Staging catalog '{catalog_name}' ready (fresh)")
        return True
    except Exception as e:
        log(f"    FAILED to create staging catalog '{catalog_name}': {str(e)[:200]}")
        return False


def ensure_install_catalog(spark, catalog_name):
    try:
        spark.sql(f"DROP CATALOG IF EXISTS `{catalog_name}` CASCADE")
        spark.sql(f"CREATE CATALOG `{catalog_name}`")
        log(f"    Install catalog '{catalog_name}' ready (fresh)")
        return True
    except Exception as e:
        log(f"    FAILED to create install catalog '{catalog_name}': {str(e)[:200]}")
        return False


def _pre_launch_validate(spark, biz_name, staging_catalog, ecm_v1_catalog, mvm_v1_catalog,
                         business, widget_values, notebook_path, effective_notebook):
    errors = []

    if not biz_name or not biz_name.strip():
        errors.append("Business name is empty or missing in the industry context file")
    if not business.get("description", "").strip():
        errors.append(f"Business '{biz_name}' has no description in the industry context file")
    if not notebook_path:
        errors.append("Notebook path is not specified")
    if not effective_notebook:
        errors.append("Effective notebook path resolved to empty")

    for key in WIDGET_VALUE_KEYS:
        if key not in widget_values:
            errors.append(f"Widget value '{key}' is missing from the context file")

    if not ensure_install_catalog(spark, ecm_v1_catalog):
        errors.append(f"Cannot create ECM install catalog '{ecm_v1_catalog}'")

    if not ensure_install_catalog(spark, mvm_v1_catalog):
        errors.append(f"Cannot create MVM install catalog '{mvm_v1_catalog}'")

    if not ensure_staging_catalog(spark, staging_catalog):
        errors.append(f"Cannot create staging catalog '{staging_catalog}'")

    if errors:
        header = (
            f"\n{'='*80}\n"
            f"❌ PRE-LAUNCH VALIDATION FAILED for '{biz_name}'\n"
            f"{'='*80}\n"
            f"\nThe following {len(errors)} issue(s) were found BEFORE launching the job.\n"
            f"Fix these issues and re-run the pipeline.\n"
        )
        detail = "\n".join(f"\n  {i+1}. {err}" for i, err in enumerate(errors))
        footer = f"\n\n{'='*80}"
        raise ValueError(header + detail + footer)

    log(f"  [Pre-launch] ✓ All validations passed")


def create_dry_run_notebook(w, runner_notebook_dir):
    dry_run_path = f"{runner_notebook_dir}/vibe_runner_dry_run"
    notebook_content = '''# Databricks notebook source
import time, json, os, re
from datetime import datetime
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

dbutils.widgets.text("business_name", "", "01. Business (name)")
dbutils.widgets.text("business_description", "", "02. Description")
dbutils.widgets.dropdown("operation", "new base model", ["new base model", "vibe modeling of version", "shrink ecm", "enlarge mvm", "install model", "uninstall model version", "generate sample data"], "03. Operation")
dbutils.widgets.dropdown("model_version", "", [""] + [str(i) for i in range(1, 101)], "04. Version")
dbutils.widgets.dropdown("data_model_scopes", "Minimum Viable Model - MVM", ["Minimum Viable Model - MVM", "Expanded Coverage Model - ECM"], "05. Model Scope")
dbutils.widgets.text("business_domains", "", "06. Business Domains")
dbutils.widgets.dropdown("org_divisions", "Operations and Business", ["Operations", "Operations and Business", "Operations, Business and Corporate"], "07. Included Org Divisions")
dbutils.widgets.text("model_vibes", "", "08. Model Vibes (inline text or /path/to/vibes.txt)")
dbutils.widgets.text("deployment_catalog", "", "09. Installation Catalog")
dbutils.widgets.dropdown("cataloging_style", "One Catalog", ["One Catalog", "Catalog per Division", "Catalog per Domain"], "09a. Cataloging Style")
dbutils.widgets.text("catalog_prefix", "", "09b. Catalog Prefix")
dbutils.widgets.text("catalog_suffix", "", "09c. Catalog Suffix")
dbutils.widgets.dropdown("generate_samples", "0", ["0", "5", "10", "15", "20", "25", "50", "100"], "10. Sample Records (0 = No Samples)")
dbutils.widgets.text("context_file", "", "11. Model JSON File (optional, e.g. /Volumes/.../model.json)")
dbutils.widgets.dropdown("naming_convention", "snake_case", ["snake_case", "camelCase", "PascalCase", "SCREAMING_CASE"], "12. Naming Convention")
dbutils.widgets.text("primary_key_suffix", "_id", "13. Primary Key Suffix")
dbutils.widgets.text("schema_prefix", "", "15. Schema Prefix (e.g. stg_, raw_, or blank)")
dbutils.widgets.text("schema_suffix", "", "15a. Schema Suffix")
dbutils.widgets.text("tag_prefix", "dbx_", "16. Tag Prefix")
dbutils.widgets.text("tag_suffix", "", "16a. Tag Suffix")
dbutils.widgets.dropdown("table_id_type", "BIGINT", ["BIGINT", "INT", "LONG", "STRING"], "17. Table ID Type")
dbutils.widgets.dropdown("boolean_format", "Boolean (True/False)", ["Boolean (True/False)", "Int (0/1)", "String (Y/N)"], "18. Boolean Format")
dbutils.widgets.dropdown("date_format", "yyyy-MM-dd", ["yyyy-MM-dd", "dd/MM/yyyy", "MM/dd/yyyy", "yyyy/MM/dd", "dd-MM-yyyy"], "19. Date Format")
dbutils.widgets.dropdown("timestamp_format", "yyyy-MM-dd\\'T\\'HH:mm:ss.SSSXXX", ["yyyy-MM-dd\\'T\\'HH:mm:ss.SSSXXX", "yyyy-MM-dd HH:mm:ss", "yyyy-MM-dd\\'T\\'HH:mm:ss", "yyyy-MM-dd HH:mm:ss.SSS"], "20. Timestamp Format")
dbutils.widgets.text("classification_levels", "restricted=restricted, confidential=confidential, internal=Internal, public=public", "21. Classification Levels (key=label pairs)")
dbutils.widgets.dropdown("housekeeping_columns", "No", ["No", "Yes"], "22. Housekeeping Columns")
dbutils.widgets.dropdown("history_tracking_columns", "No", ["No", "Yes"], "23. History Tracking Columns")
dbutils.widgets.text("vibe_session_id", "", "24. Vibe Session ID")

_WIDGET_NAMES = [
    "business_name", "business_description", "operation", "model_version",
    "data_model_scopes", "business_domains", "org_divisions", "model_vibes",
    "deployment_catalog", "cataloging_style", "catalog_prefix", "catalog_suffix",
    "generate_samples", "context_file", "naming_convention", "primary_key_suffix",
    "schema_prefix", "schema_suffix", "tag_prefix", "tag_suffix",
    "table_id_type", "boolean_format", "date_format", "timestamp_format",
    "classification_levels", "housekeeping_columns", "history_tracking_columns",
    "vibe_session_id",
]

business_name = dbutils.widgets.get("business_name")
business_description = dbutils.widgets.get("business_description")
operation = dbutils.widgets.get("operation")
deployment_catalog = dbutils.widgets.get("deployment_catalog")
data_model_scopes = dbutils.widgets.get("data_model_scopes")
context_file = dbutils.widgets.get("context_file")
model_version = dbutils.widgets.get("model_version") or "1"
generate_samples = dbutils.widgets.get("generate_samples")
vibe_session_id = dbutils.widgets.get("vibe_session_id").strip()

if not vibe_session_id:
    import uuid as _uuid
    vibe_session_id = str((_uuid.uuid4().int >> 64) & 9223372036854775807)
    print(f"  Generated session_id: {vibe_session_id}")
else:
    print(f"  Using provided session_id: {vibe_session_id}")

model_scope = "ecm" if "ECM" in data_model_scopes else "mvm"

def sanitize_name(name, strip_stop_words=True):
    if not name:
        return "unnamed_model"
    s = str(name).lower()
    if strip_stop_words:
        s = re.sub(r"[(),.&]", " ", s)
        stops = ["and","or","with","by","of","for","the","a","an","in","on","at",
                 "services","solutions","business","inc","llc","corp"]
        s = re.sub(r"\\b(" + "|".join(re.escape(w) for w in stops) + r")\\b", "", s)
    s = re.sub(r"[^a-z0-9_]", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    if s and s[0].isdigit():
        s = "_" + s
    if not s or s == "_":
        first_word = re.sub(r"[^a-z0-9]", "", str(name).lower().split()[0] if str(name).split() else "")
        return first_word if first_word else "unnamed_model"
    return s

sanitized = sanitize_name(business_name, strip_stop_words=False)
version_folder = f"v{model_version}_{model_scope}"
metamodel_db = f"`{deployment_catalog}`.`_metamodel`"
root_loc = f"/Volumes/{deployment_catalog}/_metamodel/vol_root"
target_volume = f"{root_loc}/business/{sanitized}/{version_folder}"

print("=" * 100)
print("  VIBE RUNNER DRY-RUN NOTEBOOK")
print(f"  Business:    {business_name}")
print(f"  Operation:   {operation}")
print(f"  Catalog:     {deployment_catalog}")
print(f"  Scope:       {model_scope}")
print(f"  Version:     {model_version}")
print(f"  Target Vol:  {target_volume}")
print(f"  Context:     {context_file}")
print("=" * 100)
print()

print("-" * 100)
print(f"  {'#':<4} {'WIDGET NAME':<35} {'VALUE'}")
print("-" * 100)
_empty_count = 0
for idx, wn in enumerate(_WIDGET_NAMES, 1):
    val = dbutils.widgets.get(wn)
    display_val = val if val != "" else "<EMPTY>"
    if val == "":
        _empty_count += 1
    print(f"  {idx:<4} {wn:<35} {display_val}")
print("-" * 100)
print(f"  TOTAL WIDGETS: {len(_WIDGET_NAMES)}  |  SET: {len(_WIDGET_NAMES) - _empty_count}  |  EMPTY: {_empty_count}")
print("-" * 100)
print()

FAKE_DOMAINS = [
    {"domain": "core_operations", "database_name": "core_operations", "division": "operations",
     "description": f"Core operational data for {business_name}"},
    {"domain": "customer_management", "database_name": "customer_management", "division": "business",
     "description": f"Customer management for {business_name}"},
    {"domain": "analytics", "database_name": "analytics", "division": "business",
     "description": f"Analytics and reporting for {business_name}"},
]

FAKE_PRODUCTS = [
    {"domain": "core_operations", "product": "entity_master", "type": "table",
     "primary_key": "entity_id", "table_name": "entity_master",
     "description": f"Primary entity master for {business_name}",
     "attributes": [
         {"attribute": "entity_id", "column_name": "entity_id", "type": "BIGINT"},
         {"attribute": "entity_name", "column_name": "entity_name", "type": "STRING"},
         {"attribute": "created_at", "column_name": "created_at", "type": "TIMESTAMP"},
     ]},
    {"domain": "core_operations", "product": "transaction_log", "type": "table",
     "primary_key": "transaction_id", "table_name": "transaction_log",
     "description": f"Transaction log for {business_name}",
     "attributes": [
         {"attribute": "transaction_id", "column_name": "transaction_id", "type": "BIGINT"},
         {"attribute": "entity_id", "column_name": "entity_id", "type": "BIGINT"},
         {"attribute": "amount", "column_name": "amount", "type": "DOUBLE"},
         {"attribute": "transaction_date", "column_name": "transaction_date", "type": "TIMESTAMP"},
     ]},
    {"domain": "customer_management", "product": "customer_profile", "type": "table",
     "primary_key": "customer_id", "table_name": "customer_profile",
     "description": f"Customer profiles for {business_name}",
     "attributes": [
         {"attribute": "customer_id", "column_name": "customer_id", "type": "BIGINT"},
         {"attribute": "customer_name", "column_name": "customer_name", "type": "STRING"},
         {"attribute": "segment", "column_name": "segment", "type": "STRING"},
     ]},
    {"domain": "analytics", "product": "kpi_summary", "type": "table",
     "primary_key": "kpi_id", "table_name": "kpi_summary",
     "description": f"Key performance indicators for {business_name}",
     "attributes": [
         {"attribute": "kpi_id", "column_name": "kpi_id", "type": "BIGINT"},
         {"attribute": "metric_name", "column_name": "metric_name", "type": "STRING"},
         {"attribute": "metric_value", "column_name": "metric_value", "type": "DOUBLE"},
     ]},
]

MVM_PRODUCTS = [FAKE_PRODUCTS[0], FAKE_PRODUCTS[3]]
MVM_DOMAINS = [FAKE_DOMAINS[0], FAKE_DOMAINS[2]]


def build_model_json(biz_name, biz_desc, scope, ver, domains, products):
    flat_attrs = []
    for p in products:
        for a in p.get("attributes", []):
            flat_attrs.append({
                "domain": p["domain"], "product": p["product"],
                "attribute": a["attribute"], "column_name": a["column_name"],
                "type": a["type"],
                "foreign_key_to": "", "description": f"Attribute {a['attribute']}",
            })
    return {
        "model_requirements": {
            "business_name": biz_name,
            "business_description": biz_desc,
            "model_version": ver,
            "data_model_scopes": f"Expanded Coverage Model - ECM" if scope == "ecm" else "Minimum Viable Model - MVM",
        },
        "model": {
            "name": biz_name,
            "version": f"v{ver}_{scope}",
            "description": biz_desc,
            "industry_alignment": biz_name,
            "model_scope": scope,
            "domains": domains,
            "products": products,
            "attributes": flat_attrs,
            "dry_run": True,
            "generated_at": datetime.now().isoformat(),
        }
    }


def create_metamodel_objects(catalog, scope, biz_name, biz_desc, ver, domains, products, session_id):
    mm = f"`{catalog}`.`_metamodel`"
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {mm}")
    print(f"  [META] Schema created: {mm}")

    vol_fqn = f"{mm}.`vol_root`"
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {vol_fqn}")
    print(f"  [META] Volume created: {vol_fqn}")

    spark.sql(f"""CREATE TABLE IF NOT EXISTS {mm}.business (
        business STRING, version STRING, model_scope STRING, industry_alignment STRING,
        description STRING, catalog STRING, location STRING, core_business_processes STRING,
        orgnaization_divisions STRING, data_domains STRING, common_business_jargons STRING,
        operational_systems_of_records STRING, industry_governing_body STRING,
        vibe_modelling_instructions STRING, model_conventions STRING,
        completion_date TIMESTAMP, session_id BIGINT, processing_status STRING,
        completed_percent DOUBLE, session_started_at TIMESTAMP, last_updated_at TIMESTAMP,
        session_json STRING, results_json STRING
    )""")
    spark.sql(f"""CREATE TABLE IF NOT EXISTS {mm}.domain (
        business STRING, version STRING, model_scope STRING, domain STRING,
        division STRING, description STRING, database_name STRING, catalog STRING,
        reference STRING, tags STRING
    )""")
    spark.sql(f"""CREATE TABLE IF NOT EXISTS {mm}.product (
        business STRING, version STRING, model_scope STRING, domain STRING,
        subdomain STRING, product STRING, description STRING, type STRING,
        division STRING, function STRING, data_type STRING, source_domains STRING,
        association_edges STRING, primary_key STRING, reference STRING,
        table_name STRING, sample_path STRING, tags STRING
    )""")
    spark.sql(f"""CREATE TABLE IF NOT EXISTS {mm}.attribute (
        business STRING, version STRING, model_scope STRING, domain STRING,
        product STRING, attribute STRING, column_name STRING, type STRING,
        tags STRING, value_regex STRING, foreign_key_to STRING,
        business_glossary_term STRING, description STRING, reference STRING
    )""")
    print(f"  [META] Metamodel tables created (business, domain, product, attribute)")

    _esc = lambda s: str(s).replace("'", "''")
    ver_label = f"v{ver}_{scope}"
    spark.sql(f"""INSERT INTO {mm}.business VALUES (
        '{_esc(biz_name)}', '{ver}', '{scope}', '{_esc(biz_name)}',
        '{_esc(biz_desc[:500])}', '{catalog}',
        '/Volumes/{catalog}/_metamodel/vol_root/business/{sanitize_name(biz_name, False)}/{ver_label}',
        '', '', '', '', '', '', '', '',
        current_timestamp(), {session_id}, 'completed', 100.0,
        current_timestamp(), current_timestamp(), '', ''
    )""")
    print(f"  [META] Inserted business record: {biz_name} v{ver} scope={scope} (session_id={session_id})")

    for d in domains:
        spark.sql(f"""INSERT INTO {mm}.domain VALUES (
            '{_esc(biz_name)}', '{ver}', '{scope}', '{_esc(d["domain"])}',
            '{_esc(d.get("division",""))}', '{_esc(d.get("description",""))}',
            '{_esc(d.get("database_name", d["domain"]))}', '{catalog}', '', ''
        )""")
    print(f"  [META] Inserted {len(domains)} domain records")

    for p in products:
        spark.sql(f"""INSERT INTO {mm}.product VALUES (
            '{_esc(biz_name)}', '{ver}', '{scope}', '{_esc(p["domain"])}',
            '', '{_esc(p["product"])}', '{_esc(p.get("description",""))}', '{_esc(p.get("type","table"))}',
            '', '', '', '', '', '{_esc(p.get("primary_key",""))}', '',
            '{_esc(p.get("table_name", p["product"]))}', '', ''
        )""")
    print(f"  [META] Inserted {len(products)} product records")

    attr_count = 0
    for p in products:
        for a in p.get("attributes", []):
            spark.sql(f"""INSERT INTO {mm}.attribute VALUES (
                '{_esc(biz_name)}', '{ver}', '{scope}', '{_esc(p["domain"])}',
                '{_esc(p["product"])}', '{_esc(a["attribute"])}', '{_esc(a["column_name"])}',
                '{_esc(a["type"])}', '', '', '', '', 'Attribute {_esc(a["attribute"])}', ''
            )""")
            attr_count += 1
    print(f"  [META] Inserted {attr_count} attribute records")


def create_physical_schema(catalog, domains, products):
    for d in domains:
        db_name = d.get("database_name", d["domain"])
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{db_name}`")
    print(f"  [PHYS] Created {len(domains)} schemas in {catalog}")

    table_count = 0
    for p in products:
        db_name = p["domain"]
        tbl_name = p.get("table_name", p["product"])
        cols = ", ".join([f'`{a["column_name"]}` {a["type"]}' for a in p.get("attributes", [])])
        if cols:
            spark.sql(f"CREATE TABLE IF NOT EXISTS `{catalog}`.`{db_name}`.`{tbl_name}` ({cols})")
            table_count += 1
    print(f"  [PHYS] Created {table_count} tables in {catalog}")


def write_volume_artifacts(target_vol, biz_name, scope, ver, domains, products, model_json_data):
    sql_name = sanitize_name(biz_name, strip_stop_words=False)
    ver_scope = f"{ver}_{scope}"
    subdirs = ["schemas", "metrics", "samples", "docs", "vibes", "ontology", "diagram"]
    for subdir in subdirs:
        dbutils.fs.mkdirs(f"{target_vol}/{subdir}")
    print(f"  [VOL] Created {len(subdirs)} subdirectories in {target_vol}")

    def _put(path, content):
        dbutils.fs.put(path, content, overwrite=True)

    mj_content = json.dumps(model_json_data, indent=2)
    _put(f"{target_vol}/model.json", mj_content)
    print(f"  [VOL] model.json written ({len(mj_content)} bytes)")

    for d in domains:
        dk = sanitize_name(d["domain"])
        schema_sql = f"-- Dry-run {scope.upper()} schema for domain: {d['domain']}\\n"
        db_name = d.get("database_name", d["domain"])
        schema_sql += f"CREATE SCHEMA IF NOT EXISTS `{sanitized}_{scope}`.`{db_name}`;\\n\\n"
        for p in [pp for pp in products if pp["domain"] == d["domain"]]:
            cols = ", ".join([f'`{a["column_name"]}` {a["type"]}' for a in p.get("attributes", [])])
            tbl = p.get("table_name", p["product"])
            schema_sql += f"CREATE TABLE IF NOT EXISTS `{sanitized}_{scope}`.`{db_name}`.`{tbl}` ({cols});\\n"
        _put(f"{target_vol}/schemas/{sql_name}_{dk}_schema_v{ver_scope}.sql", schema_sql)
        _put(f"{target_vol}/metrics/{sql_name}_{dk}_metrics_v{ver_scope}.sql",
             f"-- Dry-run metrics for domain: {d['domain']}\\nSELECT 1 as placeholder;\\n")
    print(f"  [VOL] Schema and metric SQL files written for {len(domains)} domains")

    _put(f"{target_vol}/schemas/{sql_name}_catalogs_v{ver_scope}.sql",
         f"CREATE CATALOG IF NOT EXISTS `{sanitized}_{scope}`;\\n")

    dbml = f"// Dry-run DBML for {biz_name} v{ver_scope}\\n"
    for p in products:
        dbml += f"Table {p.get('table_name', p['product'])} {{\\n"
        for a in p.get("attributes", []):
            dbml += f"  {a['column_name']} {a['type']}\\n"
        dbml += "}\\n\\n"
    _put(f"{target_vol}/diagram/{sql_name}_dbml_v{ver_scope}.txt", dbml)
    print(f"  [VOL] DBML diagram written")

    readme_text = f"# {biz_name} - {scope.upper()} Data Model (Dry Run)\\n"
    readme_text += f"Version: v{ver_scope}\\nGenerated: {datetime.now().isoformat()}\\n"
    readme_text += f"Domains: {len(domains)}\\nProducts: {len(products)}\\n"
    _put(f"{target_vol}/readme.md", readme_text)

    _put(f"{target_vol}/vibes/current_vibes.txt", f"Dry-run vibes for {biz_name} v{ver_scope}")
    _put(f"{target_vol}/vibes/next_vibes.txt", f"Next vibes placeholder for {biz_name}")

    _put(f"{target_vol}/ontology/{sql_name}_rdf_v{ver_scope}.ttl",
         f"@prefix : <http://example.org/{sql_name}/> .\\n# Dry-run RDF\\n")

    _put(f"{target_vol}/docs/releasenotes.txt",
         f"Release Notes - {biz_name} v{ver_scope} (Dry Run)\\n")

    csv_content = "domain,product,attribute,type,description\\n"
    for p in products:
        for a in p.get("attributes", []):
            csv_content += f"{p['domain']},{p['product']},{a['attribute']},{a['type']},dry-run\\n"
    _put(f"{target_vol}/docs/{sql_name}_model_v{ver_scope}.csv", csv_content)
    print(f"  [VOL] Docs, vibes, ontology, diagram artifacts written")

    file_count = 0
    try:
        for item in dbutils.fs.ls(target_vol):
            if item.isDir():
                file_count += len(dbutils.fs.ls(item.path))
            else:
                file_count += 1
    except Exception:
        pass
    print(f"  [VOL] Total artifacts: ~{file_count} files in {target_vol}")


print(f"\\n{'='*80}")
print(f"  EXECUTING OPERATION: {operation}")
print(f"{'='*80}\\n")

if operation == "new base model":
    use_domains = FAKE_DOMAINS if model_scope == "ecm" else MVM_DOMAINS
    use_products = FAKE_PRODUCTS if model_scope == "ecm" else MVM_PRODUCTS
    model_data = build_model_json(business_name, business_description, model_scope, model_version, use_domains, use_products)
    print(f"  Step 1: Creating metamodel objects in {deployment_catalog}...")
    create_metamodel_objects(deployment_catalog, model_scope, business_name, business_description, model_version, use_domains, use_products, vibe_session_id)
    print(f"  Step 2: Creating physical schema in {deployment_catalog}...")
    create_physical_schema(deployment_catalog, use_domains, use_products)
    print(f"  Step 3: Writing volume artifacts to {target_volume}...")
    write_volume_artifacts(target_volume, business_name, model_scope, model_version, use_domains, use_products, model_data)
    print(f"  Step 4: Simulating generation delay...")
    for i in range(1, 4):
        print(f"    hello {business_name} {i}")
        time.sleep(10)

elif operation == "install model":
    print(f"  Step 1: Reading model from context_file: {context_file}")
    model_data = None
    if context_file:
        try:
            raw = open(context_file, "r").read() if os.path.exists(context_file) else None
            if raw is None:
                from databricks.sdk import WorkspaceClient
                _w = WorkspaceClient()
                resp = _w.files.download(context_file)
                raw = resp.contents.read().decode("utf-8")
            parsed = json.loads(raw)
            model_data = parsed.get("model", parsed)
            print(f"    Model loaded: {model_data.get('name', '?')} v{model_data.get('version', '?')}")
        except Exception as e:
            print(f"    WARNING: Could not read model.json: {str(e)[:200]}")

    if model_data:
        domains = model_data.get("domains", FAKE_DOMAINS)
        products = model_data.get("products", FAKE_PRODUCTS)
    else:
        domains = FAKE_DOMAINS if model_scope == "ecm" else MVM_DOMAINS
        products = FAKE_PRODUCTS if model_scope == "ecm" else MVM_PRODUCTS
        model_data = build_model_json(business_name, business_description, model_scope, model_version, domains, products)

    print(f"  Step 2: Creating metamodel objects in {deployment_catalog}...")
    create_metamodel_objects(deployment_catalog, model_scope, business_name, business_description, model_version, domains, products, vibe_session_id)
    print(f"  Step 3: Creating physical schema in {deployment_catalog}...")
    create_physical_schema(deployment_catalog, domains, products)
    print(f"  Step 4: Simulating install delay...")
    for i in range(1, 4):
        print(f"    hello {business_name} {i}")
        time.sleep(10)

elif operation == "shrink ecm":
    print(f"  Step 1: Reading ECM model from context_file: {context_file}")
    ecm_model_data = None
    if context_file:
        try:
            raw = open(context_file, "r").read() if os.path.exists(context_file) else None
            if raw is None:
                from databricks.sdk import WorkspaceClient
                _w = WorkspaceClient()
                resp = _w.files.download(context_file)
                raw = resp.contents.read().decode("utf-8")
            parsed = json.loads(raw)
            ecm_model_data = parsed.get("model", parsed)
            print(f"    ECM model loaded: {ecm_model_data.get('name', '?')}")
        except Exception as e:
            print(f"    WARNING: Could not read ECM model: {str(e)[:200]}")

    mvm_scope = "mvm"
    mvm_version_folder = f"v{model_version}_{mvm_scope}"
    mvm_target_volume = f"{root_loc}/business/{sanitized}/{mvm_version_folder}"

    if ecm_model_data:
        ecm_domains = ecm_model_data.get("domains", FAKE_DOMAINS)
        ecm_products = ecm_model_data.get("products", FAKE_PRODUCTS)
        mvm_domains = ecm_domains[:2] if len(ecm_domains) > 2 else ecm_domains
        mvm_products = [p for p in ecm_products if p.get("domain") in [d["domain"] for d in mvm_domains]]
        if not mvm_products:
            mvm_products = ecm_products[:2]
    else:
        mvm_domains = MVM_DOMAINS
        mvm_products = MVM_PRODUCTS

    mvm_model_data = build_model_json(business_name, business_description, mvm_scope, model_version, mvm_domains, mvm_products)
    print(f"  Step 2: Creating metamodel for MVM in {deployment_catalog}...")
    create_metamodel_objects(deployment_catalog, mvm_scope, business_name, business_description, model_version, mvm_domains, mvm_products, vibe_session_id)
    print(f"  Step 3: Writing MVM artifacts to {mvm_target_volume}...")
    write_volume_artifacts(mvm_target_volume, business_name, mvm_scope, model_version, mvm_domains, mvm_products, mvm_model_data)
    print(f"  Step 4: Simulating shrink delay...")
    for i in range(1, 4):
        print(f"    hello {business_name} {i}")
        time.sleep(10)

else:
    print(f"  Unsupported operation for dry run: {operation}")
    for i in range(1, 4):
        print(f"    hello {business_name} {i}")
        time.sleep(10)

print()
print(f"  Completed: {operation} for {business_name}")
print("=" * 100)
dbutils.notebook.exit(f"DRY_RUN_OK: {business_name} - {operation}")
'''
    import base64
    encoded = base64.b64encode(notebook_content.encode('utf-8')).decode('utf-8')
    try:
        w.workspace.import_(
            path=dry_run_path,
            content=encoded,
            format=ImportFormat.SOURCE,
            language=Language.PYTHON,
            overwrite=True
        )
        log(f"  Created dry-run notebook: {dry_run_path}")
    except Exception as e:
        log(f"  WARNING: Could not create dry-run notebook: {str(e)[:200]}")
        log(f"  Falling back to using main notebook path")
        return None
    return dry_run_path


def _get_workspace_host_and_org():
    try:
        import json as _ctx_json
        _ctx_raw = (
            dbutils.notebook.entry_point
            .getDbutils().notebook().getContext().toJson()
        )
        _ctx = _ctx_json.loads(_ctx_raw)
        _host = _ctx.get("extraContext", {}).get("api_url", "")
        _org = _ctx.get("extraContext", {}).get("orgId", "")
        if _host:
            return _host.rstrip("/"), _org
    except Exception:
        pass
    try:
        _w = WorkspaceClient()
        return str(_w.config.host).rstrip("/"), ""
    except Exception:
        pass
    return "", ""


def submit_notebook_run(w, notebook_path, params, run_name, timeout_seconds=JOB_TIMEOUT_SECONDS,
                        business_name="", operation="", model_scope="ecm", version="1"):
    try:
        biz = business_name or params.get("business_name", "unknown")
        op = operation or params.get("operation", "unknown")
        scope = model_scope or "ecm"
        ver = version or params.get("model_version", "1") or "1"
        sid = params.get("vibe_session_id", "")

        job_tags = build_job_tags(biz, op, notebook_path, model_scope=scope, version=ver,
                                  session_id=sid)
        _san_op = sanitize_name(op, strip_stop_words=False) if op else "run"
        _san_biz = sanitize_name(biz, strip_stop_words=False) if biz else "unknown"
        job_name = run_name or f"dbx_vibe_{_san_biz}_{_san_op}_{scope}_v{ver}"

        job_id, run_id, reused = find_or_create_job(
            w, job_name, notebook_path, params, job_tags,
            timeout_seconds=timeout_seconds,
        )
        log(f"    Submitted: {run_name} (job_id={job_id}, run_id={run_id}, reused={reused})")
        return run_id
    except Exception as e:
        log(f"    FAILED to submit run '{run_name}': {str(e)[:300]}")
        return None


def wait_for_run(w, run_id, run_name, job_id=None, poll_interval=POLL_INTERVAL_SECONDS):
    start_time = time.time()
    log(f"    Monitoring run {run_id} ({run_name})...")
    url_printed = False
    last_log_interval = -1
    while True:
        try:
            run = w.jobs.get_run(run_id=run_id)
            state = run.state.life_cycle_state.value
            elapsed = (time.time() - start_time)
            elapsed_str = f"{elapsed / 3600:.2f}h" if elapsed > 3600 else f"{elapsed / 60:.1f}m"

            if not url_printed:
                page_url = getattr(run, "run_page_url", None) or ""
                if not page_url:
                    host, org_id = _get_workspace_host_and_org()
                    if host:
                        org_param = f"?o={org_id}" if org_id else ""
                        jid = job_id or 0
                        page_url = f"{host}/jobs/{jid}/runs/{run_id}{org_param}"
                if page_url:
                    log(f"    URL: {page_url}")
                url_printed = True

            if state in ('TERMINATED', 'SKIPPED', 'INTERNAL_ERROR'):
                result_state = run.state.result_state.value if run.state.result_state else 'UNKNOWN'
                duration_hrs = elapsed / 3600
                if result_state == 'SUCCESS':
                    log(f"    SUCCESS: {run_name} ({elapsed_str})")
                else:
                    error_msg = ""
                    if run.state.state_message:
                        error_msg = f" - {run.state.state_message[:150]}"
                    log(f"    FAILED: {run_name} -> {result_state} ({elapsed_str}){error_msg}")
                return result_state, duration_hrs

            current_interval = int(elapsed / (poll_interval * 10))
            if current_interval > last_log_interval:
                last_log_interval = current_interval
                log(f"      {run_name}: {state} ({elapsed_str})")

        except Exception as e:
            log(f"      Error polling run {run_id}: {str(e)[:100]}")

        time.sleep(poll_interval)


def _parse_exit_json(raw_value, default_status="unknown"):
    """Parse structured exit JSON from a notebook run. Returns (status, warnings_list).
    Handles None, non-JSON, non-dict JSON, null warnings, non-list warnings, non-string items."""
    if not raw_value:
        return default_status, []
    try:
        parsed = json.loads(raw_value)
    except (json.JSONDecodeError, ValueError, TypeError):
        return default_status, []
    if not isinstance(parsed, dict):
        return default_status, []
    status = parsed.get("status", default_status)
    raw_warnings = parsed.get("warnings")
    warnings = [str(w) for w in raw_warnings] if isinstance(raw_warnings, list) else []
    return status, warnings


def wait_for_multi_task_run(w, run_id, job_name, task_keys,
                           biz_name="", biz_idx=0, biz_total=0, ping_interval=60,
                           job_id=None, poll_interval=POLL_INTERVAL_SECONDS):
    start_time = time.time()
    biz_label = f"{biz_name} - {biz_idx}/{biz_total}" if biz_total else job_name
    log(f"    Monitoring job {run_id} ({job_name})...")
    url_printed = False
    last_ping_time = 0
    reported_tasks = set()

    while True:
        try:
            run = w.jobs.get_run(run_id=run_id)
            state = run.state.life_cycle_state.value
            elapsed = (time.time() - start_time)
            elapsed_str = f"{elapsed / 3600:.2f}h" if elapsed > 3600 else f"{elapsed / 60:.1f}m"

            if not url_printed:
                page_url = getattr(run, "run_page_url", None) or ""
                if not page_url:
                    host, org_id = _get_workspace_host_and_org()
                    if host:
                        org_param = f"?o={org_id}" if org_id else ""
                        jid = job_id or 0
                        page_url = f"{host}/jobs/{jid}/runs/{run_id}{org_param}"
                if page_url:
                    log(f"    URL: {page_url}")
                url_printed = True

            if hasattr(run, 'tasks') and run.tasks:
                for task in run.tasks:
                    tk = task.task_key
                    if tk in reported_tasks:
                        continue
                    if task.state and task.state.life_cycle_state:
                        lcs = task.state.life_cycle_state.value
                        if lcs in ('TERMINATED', 'SKIPPED', 'INTERNAL_ERROR'):
                            rs = "SKIPPED"
                            if task.state.result_state:
                                rs = task.state.result_state.value
                            dur = ""
                            try:
                                if task.start_time and task.end_time:
                                    dur = f" ({(task.end_time - task.start_time) / 3600000:.2f}h)"
                            except Exception:
                                pass
                            status_icon = "OK" if rs == "SUCCESS" else "SKIP" if rs == "SKIPPED" else "FAIL"
                            log(f"      [{status_icon}] {tk}: {rs}{dur}")
                            reported_tasks.add(tk)

            if state in ('TERMINATED', 'SKIPPED', 'INTERNAL_ERROR'):
                overall_result = run.state.result_state.value if run.state.result_state else 'UNKNOWN'
                total_hrs = elapsed / 3600

                task_results = {}
                if hasattr(run, 'tasks') and run.tasks:
                    for task in run.tasks:
                        tk = task.task_key
                        t_state = "UNKNOWN"
                        t_hrs = 0.0
                        if task.state:
                            if task.state.result_state:
                                t_state = task.state.result_state.value
                            elif task.state.life_cycle_state and task.state.life_cycle_state.value == "SKIPPED":
                                t_state = "SKIPPED"
                        try:
                            if task.start_time and task.end_time:
                                t_hrs = (task.end_time - task.start_time) / 3600000
                        except Exception:
                            pass
                        _task_warnings = []
                        _task_exit_status = t_state.lower() if t_state else "unknown"
                        if t_state == "SUCCESS" and hasattr(task, 'run_id') and task.run_id:
                            _exit_val = None
                            try:
                                _task_output = w.jobs.get_run_output(run_id=task.run_id)
                                if _task_output and hasattr(_task_output, 'notebook_output') and _task_output.notebook_output:
                                    _exit_val = _task_output.notebook_output.result
                            except Exception as _output_err:
                                log(f"      ℹ️  {tk}: Could not read task output: {str(_output_err)[:80]}")
                                try:
                                    _task_run_detail = w.jobs.get_run(run_id=task.run_id)
                                    if hasattr(_task_run_detail, 'output') and _task_run_detail.output:
                                        _nb_out = getattr(_task_run_detail.output, 'notebook_output', None)
                                        if _nb_out:
                                            _exit_val = _nb_out.result
                                except Exception as _fallback_err:
                                    log(f"      ℹ️  {tk}: Fallback get_run also failed: {str(_fallback_err)[:80]}")
                            _parsed_status, _parsed_warnings = _parse_exit_json(_exit_val, t_state.lower())
                            _task_exit_status = _parsed_status
                            _task_warnings = _parsed_warnings
                            if _task_warnings:
                                log(f"      ⚠️  {tk}: Completed with {len(_task_warnings)} warning(s):")
                                for _tw in _task_warnings:
                                    log(f"         - {_tw.strip()[:120]}")
                        task_results[tk] = {"state": t_state, "duration_hrs": t_hrs, "warnings": _task_warnings, "exit_status": _task_exit_status}

                for tk in task_keys:
                    if tk not in task_results:
                        task_results[tk] = {"state": "UNKNOWN", "duration_hrs": 0.0, "warnings": [], "exit_status": "unknown"}

                log(f"    Job {job_name}: {overall_result} (total {elapsed_str})")
                return task_results, total_hrs

            now = time.time()
            if (now - last_ping_time) >= ping_interval:
                last_ping_time = now
                active = []
                done_count = len(reported_tasks)
                if hasattr(run, 'tasks') and run.tasks:
                    active = [t.task_key for t in run.tasks
                              if t.state and t.state.life_cycle_state
                              and t.state.life_cycle_state.value == "RUNNING"]
                active_str = f" - active: {', '.join(active)}" if active else ""
                log(f"    {biz_label} - Ping {run_id} success - still alive ({elapsed_str}, {done_count}/{len(task_keys)} tasks done){active_str}")

        except Exception as e:
            now = time.time()
            if (now - last_ping_time) >= ping_interval:
                last_ping_time = now
                log(f"    {biz_label} - Ping {run_id} error - {str(e)[:80]}")

        time.sleep(poll_interval)


def get_model_json_path(catalog_name, business_name, version, scope):
    sanitized = sanitize_name(business_name, strip_stop_words=False)
    return f"/Volumes/{catalog_name}/_metamodel/vol_root/business/{sanitized}/v{version}_{scope}/model.json"


def get_business_volume_root(catalog_name, business_name):
    sanitized = sanitize_name(business_name, strip_stop_words=False)
    return f"/Volumes/{catalog_name}/_metamodel/vol_root/business/{sanitized}"


def copy_business_folder(source_catalog, business_name, folder_path):
    sanitized = sanitize_name(business_name, strip_stop_words=False)
    source_base = get_business_volume_root(source_catalog, business_name)
    dest_base = os.path.join(folder_path, sanitized)

    try:
        if os.path.exists(source_base):
            shutil.copytree(source_base, dest_base, dirs_exist_ok=True)
            file_count = sum(len(files) for _, _, files in os.walk(dest_base))
            log(f"    Copied {source_base} -> {dest_base} ({file_count} files)")
            return True
        else:
            dbutils.fs.cp(source_base, dest_base, recurse=True)
            log(f"    Copied {source_base} -> {dest_base} (via dbutils)")
            return True
    except Exception as e:
        log(f"    FAILED to copy folder: {str(e)[:200]}")
        return False


def drop_catalog(spark, catalog_name):
    try:
        spark.sql(f"DROP CATALOG IF EXISTS `{catalog_name}` CASCADE")
        log(f"    Dropped catalog: {catalog_name}")
        return True
    except Exception as e:
        log(f"    FAILED to drop catalog '{catalog_name}': {str(e)[:200]}")
        return False


def verify_copied_files(folder_path, business_name):
    sanitized = sanitize_name(business_name, strip_stop_words=False)
    dest_base = os.path.join(folder_path, sanitized)

    if not os.path.exists(dest_base):
        log(f"    VERIFY FAILED: {dest_base} does not exist")
        return False

    file_count = 0
    dir_count = 0
    for root, dirs, files in os.walk(dest_base):
        dir_count += len(dirs)
        file_count += len(files)

    if file_count == 0:
        log(f"    VERIFY FAILED: {dest_base} has no files")
        return False

    model_json_found = False
    for root, dirs, files in os.walk(dest_base):
        if "model.json" in files:
            model_json_found = True
            break

    log(f"    VERIFY OK: {dest_base} ({file_count} files, {dir_count} dirs, model.json={'found' if model_json_found else 'MISSING'})")
    return True


def run_pipeline_for_business(w, spark, business, widget_values, notebook_path, folder_path,
                              dry_run, dry_run_notebook_path, results,
                              biz_idx=0, biz_total=0, ping_interval=60):
    biz_name = business["name"]
    sanitized = sanitize_name(biz_name, strip_stop_words=False)

    staging_catalog = f"{sanitized}_ecm"
    ecm_v1_catalog = f"{sanitized}_ecm_v1"
    mvm_v1_catalog = f"{sanitized}_mvm_v1"

    effective_notebook = dry_run_notebook_path if dry_run and dry_run_notebook_path else notebook_path
    version = "1"
    job_name = f"dbx_vibe_{sanitized}_pipeline_ecm_mvm_v{version}"

    result = {
        "industry": biz_name, "timestamp": "",
        "ecm_generated": "N/A", "ecm_installed": "N/A", "ecm_copied": "N/A",
        "ecm_catalog": ecm_v1_catalog, "ecm_total_hrs": 0.0,
        "mvm_generated": "N/A", "mvm_installed": "N/A",
        "mvm_catalog": mvm_v1_catalog, "mvm_total_hrs": 0.0,
        "warning_count": 0, "warnings": [],
    }

    tk_ecm_gen = f"{sanitized}_ecm_generate"
    tk_ecm_inst = f"{sanitized}_ecm_install_v1"
    tk_ecm_uninstall = f"{sanitized}_ecm_uninstall_staging"
    tk_mvm_shrink = f"{sanitized}_mvm_shrink"
    tk_mvm_inst = f"{sanitized}_mvm_install_v1"
    all_task_keys = [tk_ecm_gen, tk_ecm_inst, tk_ecm_uninstall, tk_mvm_shrink, tk_mvm_inst]

    log(f"{'='*80}")
    log(f"  INDUSTRY: {biz_name}")
    log(f"  Job: {job_name}")
    log(f"  Catalogs:")
    log(f"    Staging           : {staging_catalog}  (ECM+MVM workspace, dropped after installs)")
    log(f"    ECM install       : {ecm_v1_catalog}  (dedicated catalog, no schema prefix)")
    log(f"    MVM install       : {mvm_v1_catalog}  (dedicated catalog, no schema prefix)")
    log(f"  Task DAG (sequential):")
    log(f"    {tk_ecm_gen}")
    log(f"    └── {tk_ecm_inst}")
    log(f"        └── {tk_ecm_uninstall}")
    log(f"            └── {tk_mvm_shrink}")
    log(f"                └── {tk_mvm_inst}")
    log(f"{'='*80}")
    log()
    log(f"  [Pre-launch] Validating catalogs, parameters, and detecting clashes...")
    try:
        _pre_launch_validate(
            spark, biz_name, staging_catalog,
            ecm_v1_catalog, mvm_v1_catalog,
            business, widget_values, notebook_path, effective_notebook,
        )
    except ValueError as e:
        log(str(e))
        drop_catalog(spark, staging_catalog)
        result["ecm_generated"] = "failed"
        result["timestamp"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        results.append(result)
        return False

    ecm_model_json = get_model_json_path(staging_catalog, biz_name, version, "ecm")
    mvm_model_json = get_model_json_path(staging_catalog, biz_name, version, "mvm")

    ecm_gen_params = build_notebook_params(
        widget_values, business,
        operation="new base model",
        deployment_catalog=staging_catalog,
        data_model_scopes="Expanded Coverage Model - ECM",
        generate_samples=0,
    )
    ecm_gen_params["schema_prefix"] = ""

    ecm_install_params = build_notebook_params(
        widget_values, business,
        operation="install model",
        deployment_catalog=ecm_v1_catalog,
        data_model_scopes="Expanded Coverage Model - ECM",
        context_file=ecm_model_json,
        model_version=version,
        generate_samples=0,
    )
    ecm_install_params["schema_prefix"] = ""

    ecm_uninstall_params = build_notebook_params(
        widget_values, business,
        operation="uninstall model version",
        deployment_catalog=staging_catalog,
        data_model_scopes="Expanded Coverage Model - ECM",
        model_version=version,
        generate_samples=0,
    )
    ecm_uninstall_params["schema_prefix"] = ""

    shrink_params = build_notebook_params(
        widget_values, business,
        operation="shrink ecm",
        deployment_catalog=staging_catalog,
        data_model_scopes="Expanded Coverage Model - ECM",
        context_file=ecm_model_json,
        generate_samples=0,
        model_version=version,
    )
    shrink_params["schema_prefix"] = ""

    mvm_install_params = build_notebook_params(
        widget_values, business,
        operation="install model",
        deployment_catalog=mvm_v1_catalog,
        data_model_scopes="Minimum Viable Model - MVM",
        context_file=mvm_model_json,
        model_version=version,
        generate_samples=0,
    )
    mvm_install_params["schema_prefix"] = ""

    pipeline_session_id = generate_session_id()
    log(f"  Pipeline session_id: {pipeline_session_id}")
    log(f"  Creating job: {job_name} (5 tasks)")
    try:
        job_tags = build_job_tags(
            business_name=biz_name,
            operation="new base model",
            notebook_path=effective_notebook,
            model_scope="ecm",
            version=version,
            session_id=pipeline_session_id,
        )

        task_configs = [
            {
                "task_key": tk_ecm_gen,
                "notebook_path": effective_notebook,
                "params": ecm_gen_params,
                "timeout_seconds": JOB_TIMEOUT_SECONDS,
            },
            {
                "task_key": tk_ecm_inst,
                "notebook_path": effective_notebook,
                "params": ecm_install_params,
                "timeout_seconds": JOB_TIMEOUT_SECONDS,
                "depends_on": [tk_ecm_gen],
            },
            {
                "task_key": tk_ecm_uninstall,
                "notebook_path": effective_notebook,
                "params": ecm_uninstall_params,
                "timeout_seconds": JOB_TIMEOUT_SECONDS,
                "depends_on": [tk_ecm_inst],
            },
            {
                "task_key": tk_mvm_shrink,
                "notebook_path": effective_notebook,
                "params": shrink_params,
                "timeout_seconds": JOB_TIMEOUT_SECONDS,
                "depends_on": [tk_ecm_uninstall],
            },
            {
                "task_key": tk_mvm_inst,
                "notebook_path": effective_notebook,
                "params": mvm_install_params,
                "timeout_seconds": JOB_TIMEOUT_SECONDS,
                "depends_on": [tk_mvm_shrink],
            },
        ]

        job_id, run_id, reused = find_or_create_job(
            w, job_name, effective_notebook, {}, job_tags,
            task_configs=task_configs,
            timeout_seconds=JOB_TIMEOUT_SECONDS,
        )
    except Exception as e:
        log(f"    FAILED to create/run job '{job_name}': {str(e)[:300]}")
        drop_catalog(spark, staging_catalog)
        result["ecm_generated"] = "failed"
        result["timestamp"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        results.append(result)
        return False

    task_results, total_hrs = wait_for_multi_task_run(
        w, run_id, job_name, all_task_keys,
        biz_name=biz_name, biz_idx=biz_idx, biz_total=biz_total, ping_interval=ping_interval,
        job_id=job_id,
    )

    ecm_gen_state = task_results.get(tk_ecm_gen, {}).get("state", "UNKNOWN")
    ecm_inst_state = task_results.get(tk_ecm_inst, {}).get("state", "UNKNOWN")
    ecm_uninstall_state = task_results.get(tk_ecm_uninstall, {}).get("state", "UNKNOWN")
    mvm_shrink_state = task_results.get(tk_mvm_shrink, {}).get("state", "UNKNOWN")
    mvm_inst_state = task_results.get(tk_mvm_inst, {}).get("state", "UNKNOWN")

    ecm_gen_ok = ecm_gen_state == "SUCCESS"
    ecm_install_ok = ecm_inst_state == "SUCCESS"
    ecm_uninstall_ok = ecm_uninstall_state == "SUCCESS"
    mvm_gen_ok = mvm_shrink_state == "SUCCESS"
    mvm_install_ok = mvm_inst_state == "SUCCESS"

    ecm_gen_hrs = task_results.get(tk_ecm_gen, {}).get("duration_hrs", 0.0)
    ecm_inst_hrs = task_results.get(tk_ecm_inst, {}).get("duration_hrs", 0.0)
    mvm_shrink_hrs = task_results.get(tk_mvm_shrink, {}).get("duration_hrs", 0.0)
    mvm_inst_hrs = task_results.get(tk_mvm_inst, {}).get("duration_hrs", 0.0)

    ecm_inst_exit = task_results.get(tk_ecm_inst, {}).get("exit_status", "")
    mvm_inst_exit = task_results.get(tk_mvm_inst, {}).get("exit_status", "")
    ecm_gen_exit = task_results.get(tk_ecm_gen, {}).get("exit_status", "")
    mvm_shrink_exit = task_results.get(tk_mvm_shrink, {}).get("exit_status", "")
    ecm_inst_warnings = task_results.get(tk_ecm_inst, {}).get("warnings") or []
    mvm_inst_warnings = task_results.get(tk_mvm_inst, {}).get("warnings") or []
    ecm_gen_warnings = task_results.get(tk_ecm_gen, {}).get("warnings") or []
    mvm_shrink_warnings = task_results.get(tk_mvm_shrink, {}).get("warnings") or []
    all_warnings = list(ecm_gen_warnings) + list(ecm_inst_warnings) + list(mvm_shrink_warnings) + list(mvm_inst_warnings)

    def _status_with_warnings(ok, exit_status, warnings, na_condition=False, na_label="N/A"):
        if na_condition:
            return na_label
        if not ok:
            return "failed"
        if warnings or exit_status == "success_with_warnings":
            return "warning"
        return "success"

    result["ecm_generated"] = _status_with_warnings(ecm_gen_ok, ecm_gen_exit, ecm_gen_warnings)
    result["ecm_installed"] = _status_with_warnings(ecm_install_ok, ecm_inst_exit, ecm_inst_warnings, na_condition=not ecm_gen_ok)
    result["ecm_total_hrs"] = round(ecm_gen_hrs + ecm_inst_hrs, 2)

    result["mvm_generated"] = _status_with_warnings(mvm_gen_ok, mvm_shrink_exit, mvm_shrink_warnings, na_condition=not ecm_gen_ok)
    result["mvm_installed"] = _status_with_warnings(mvm_install_ok, mvm_inst_exit, mvm_inst_warnings, na_condition=not mvm_gen_ok)
    result["mvm_total_hrs"] = round(mvm_shrink_hrs + mvm_inst_hrs, 2)

    result["warning_count"] = len(all_warnings)
    result["warnings"] = all_warnings

    all_ok = ecm_gen_ok and ecm_install_ok and ecm_uninstall_ok and mvm_gen_ok and mvm_install_ok
    copy_ok = False
    if all_ok:
        log(f"  Copying {biz_name} folder to {folder_path}...")
        copy_ok = copy_business_folder(staging_catalog, biz_name, folder_path)
        if copy_ok:
            verify_copied_files(folder_path, biz_name)
    else:
        log(f"  SKIPPED folder copy — not all tasks succeeded")
        failed_parts = []
        if not ecm_gen_ok:
            failed_parts.append("ECM generation")
        if not ecm_install_ok:
            failed_parts.append(f"ECM install ({ecm_v1_catalog})")
        if not ecm_uninstall_ok:
            failed_parts.append(f"ECM uninstall from staging ({staging_catalog})")
        if not mvm_gen_ok:
            failed_parts.append("MVM shrink")
        if not mvm_install_ok:
            failed_parts.append(f"MVM install ({mvm_v1_catalog})")
        log(f"    Failed: {', '.join(failed_parts)}")

    log(f"  Dropping staging catalog...")
    drop_catalog(spark, staging_catalog)

    result["ecm_copied"] = "success" if copy_ok else ("N/A" if not all_ok else "failed")
    result["timestamp"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    if all_warnings:
        log(f"  ⚠️  {biz_name}: {len(all_warnings)} warning(s) detected across tasks — marked as 'warning' in results")

    results.append(result)
    return all_ok


def save_results(results, folder_path):
    pass


def display_results(results):
    log(f"{'='*160}")
    log("  VIBE RUNNER RESULTS SUMMARY")
    log(f"{'='*160}")
    log(f"  {'Industry':<22} {'Timestamp':<21} {'ECM Gen':<10} {'ECM Inst':<10} {'ECM Copy':<10} {'ECM Catalog':<28} {'ECM Hrs':<9} {'MVM Gen':<10} {'MVM Inst':<10} {'MVM Catalog':<28} {'MVM Hrs':<9}")
    log(f"  {'-'*22} {'-'*21} {'-'*10} {'-'*10} {'-'*10} {'-'*28} {'-'*9} {'-'*10} {'-'*10} {'-'*28} {'-'*9}")

    for r in results:
        log(f"  {r.get('industry','?'):<22} {r.get('timestamp','?'):<21} {r.get('ecm_generated','?'):<10} {r.get('ecm_installed','?'):<10} {r.get('ecm_copied','?'):<10} {r.get('ecm_catalog','?'):<28} {r.get('ecm_total_hrs',0):<9} {r.get('mvm_generated','?'):<10} {r.get('mvm_installed','?'):<10} {r.get('mvm_catalog','?'):<28} {r.get('mvm_total_hrs',0):<9}")

    total = len(results)
    ecm_gen_ok = sum(1 for r in results if r.get('ecm_generated') in ('success', 'warning'))
    ecm_inst_ok = sum(1 for r in results if r.get('ecm_installed') in ('success', 'warning'))
    ecm_copy_ok = sum(1 for r in results if r.get('ecm_copied') == 'success')
    mvm_gen_ok = sum(1 for r in results if r.get('mvm_generated') in ('success', 'warning'))
    mvm_inst_ok = sum(1 for r in results if r.get('mvm_installed') in ('success', 'warning'))
    total_warnings = sum(r.get('warning_count', 0) for r in results)
    ecm_gen_warned = sum(1 for r in results if r.get('ecm_generated') == 'warning')
    ecm_inst_warned = sum(1 for r in results if r.get('ecm_installed') == 'warning')
    mvm_gen_warned = sum(1 for r in results if r.get('mvm_generated') == 'warning')
    mvm_inst_warned = sum(1 for r in results if r.get('mvm_installed') == 'warning')
    total_hrs = sum(r.get('ecm_total_hrs', 0) + r.get('mvm_total_hrs', 0) for r in results)

    log(f"  TOTALS:")
    log(f"    Industries processed : {total}")
    log(f"    ECM generated        : {ecm_gen_ok}/{total}" + (f"  ({ecm_gen_warned} with warnings)" if ecm_gen_warned else ""))
    log(f"    ECM installed        : {ecm_inst_ok}/{total}" + (f"  ({ecm_inst_warned} with warnings)" if ecm_inst_warned else ""))
    log(f"    ECM copied           : {ecm_copy_ok}/{total}")
    log(f"    MVM generated        : {mvm_gen_ok}/{total}" + (f"  ({mvm_gen_warned} with warnings)" if mvm_gen_warned else ""))
    log(f"    MVM installed        : {mvm_inst_ok}/{total}" + (f"  ({mvm_inst_warned} with warnings)" if mvm_inst_warned else ""))
    log(f"    Total runtime        : {total_hrs:.2f} hours")
    if total_warnings > 0:
        log(f"    ⚠️  Total warnings    : {total_warnings}")
        for r in results:
            r_warnings = r.get("warnings") or []
            if r_warnings:
                log(f"      {r.get('industry', 'unknown')}:")
                for _rw in r_warnings:
                    log(f"        - {str(_rw).strip()[:120]}")
    log(f"{'='*160}")


def main():
    print(RUNNER_ASCII_ART)

    pipeline_start = time.time()

    create_widgets()
    config = read_widgets()

    business_context_path = config["business_context_path"]
    notebook_path = config["notebook_path"]
    folder_path = config["folder_path"]
    dry_run = config["dry_run"]
    ping_interval = config["ping_interval"]

    log(f"{'='*80}")
    log(f"  LOADING INDUSTRY CONTEXT")
    log(f"{'='*80}")
    log()
    widget_values, businesses = load_business_context(business_context_path)

    w = WorkspaceClient()
    spark = SparkSession.builder.getOrCreate()
    log(f"  Databricks client initialized")
    log(f"  Spark session ready")

    dry_run_notebook_path = None
    if dry_run:
        log(f"  DRY RUN MODE ENABLED")
        log(f"  Creating dry-run dummy notebook...")
        try:
            current_notebook = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
            runner_dir = os.path.dirname(current_notebook)
        except Exception:
            runner_dir = os.path.dirname(notebook_path)
        dry_run_notebook_path = create_dry_run_notebook(w, runner_dir)

    os.makedirs(folder_path, exist_ok=True)

    results = []
    total = len(businesses)

    log(f"{'='*80}")
    log(f"  STARTING PIPELINE: {total} industries")
    log(f"  Mode: {'DRY RUN' if dry_run else 'PRODUCTION'}")
    log(f"{'='*80}")

    for idx, business in enumerate(businesses, 1):
        biz_name = business["name"]
        log(f"{'#'*80}")
        log(f"  [{idx}/{total}] Processing: {biz_name}")
        log(f"{'#'*80}")

        biz_start = time.time()
        try:
            success = run_pipeline_for_business(
                w=w,
                spark=spark,
                business=business,
                widget_values=widget_values,
                notebook_path=notebook_path,
                folder_path=folder_path,
                dry_run=dry_run,
                dry_run_notebook_path=dry_run_notebook_path,
                results=results,
                biz_idx=idx,
                biz_total=total,
                ping_interval=ping_interval,
            )
            biz_elapsed = (time.time() - biz_start) / 3600
            status = "ALL OK" if success else "PARTIAL/FAILED"
            log(f"  [{idx}/{total}] {biz_name}: {status} ({biz_elapsed:.2f} hrs)")
        except Exception as e:
            biz_elapsed = (time.time() - biz_start) / 3600
            log(f"  [{idx}/{total}] {biz_name}: EXCEPTION ({biz_elapsed:.2f} hrs)")
            log(f"    Error: {str(e)[:300]}")
            already_tracked = any(r.get('industry') == biz_name for r in results)
            if not already_tracked:
                sanitized = sanitize_name(biz_name, strip_stop_words=False)
                results.append({
                    "industry": biz_name, "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                    "ecm_generated": "exception", "ecm_installed": "N/A", "ecm_copied": "N/A",
                    "ecm_catalog": f"{sanitized}_ecm_v1", "ecm_total_hrs": round(biz_elapsed, 2),
                    "mvm_generated": "N/A", "mvm_installed": "N/A",
                    "mvm_catalog": f"{sanitized}_mvm_v1", "mvm_total_hrs": 0.0,
                    "warning_count": 0, "warnings": [],
                })

    pipeline_elapsed = (time.time() - pipeline_start) / 3600

    log(f"{'='*80}")
    log(f"  PIPELINE COMPLETE")
    log(f"  Total runtime: {pipeline_elapsed:.2f} hours")
    log(f"{'='*80}")

    display_results(results)
    results_file = save_results(results, folder_path)

    try:
        import pandas as pd
        df = pd.DataFrame(results)
        display(df)
    except Exception:
        pass

    _PASS_STATUSES = ("success", "warning")
    failed_businesses = [
        r.get("industry", "unknown") for r in results
        if r.get("ecm_generated") not in _PASS_STATUSES
        or r.get("ecm_installed") not in (*_PASS_STATUSES, "N/A")
        or r.get("mvm_generated") not in (*_PASS_STATUSES, "N/A")
        or r.get("mvm_installed") not in (*_PASS_STATUSES, "N/A")
    ]
    if failed_businesses:
        failure_summary = ", ".join(failed_businesses)
        raise RuntimeError(
            f"Pipeline FAILED for {len(failed_businesses)}/{len(results)} "
            f"industry(ies): {failure_summary}. "
            f"Check the results table above for details."
        )

    return results


create_widgets()

In [0]:
# ============================================================================
# RUN MAIN
# ============================================================================

if __name__ == "__main__":
    jobs_result = main() 
    #create_widgets()